[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/multiple_regression.ipynb)

# Multiple regression full course: the practice set, in Python

Thirty-four exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (fitting with several x's, R² and the adjusted R², the F and t tests, the two intervals,
residuals, logistic regression, a squared term, the partial F test, the Durbin–Watson test), **tier 3** stretch
problems (combine ideas, spot the trap, work backwards) and **tier 4**, the case: 240 vans sold at auction by Corrowby
Fleet Leasing (a fictional company; our own data), and the question of which vans are worth a $1,500 refurbishment
before they're sold. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same models with `numpy`, `scipy.stats`,
`pandas` and, where it's installed, `statsmodels`, so you can run them on your own data. That includes the logistic
regressions, which no single spreadsheet function can fit. And a few cells you can move yourself: a flat's size and
distance, with its two intervals; and a borrower's debt-to-income ratio on the S-curve.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Decimals as
decimals: `0.107`, not `10.7%`; a p-value to four decimals is close enough. Numbers without thousands separators:
`23183`, not `23,183`. You can type the arithmetic too: `22+3.5*10-1.2*5`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). You can also put your answer between
the quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[multiple regression cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/multiple_regression_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the checker and the fitting functions: run this cell first.
import ast, math, operator, random, re, textwrap
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox, Text, Button, Output, HBox, VBox, Layout
try:                                                  # statsmodels: in the browser version and in Colab
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.stats.stattools import durbin_watson as sm_durbin_watson
except ImportError:                                   # a Python without it: the numpy functions below do the same job
    sm = None

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the data CHALK · the fitted line or plane SKY · residuals VIOLET · a dummy's group MINT · a misleading analysis CORAL ·
# the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- least squares and logistic regression in plain numpy (what LINEST and statsmodels do inside) ----
def ols(y, X):
    """Least squares of y on the columns of X, with an intercept: the b's, their standard errors, t and p, R², the
    adjusted R², s, F and its p-value (everything LINEST gives), plus ŷ, residuals, (X′X)⁻¹ and each row's leverage."""
    y = np.asarray(y, float); X1 = np.column_stack([np.ones(len(y)), np.asarray(X, float)])
    n, k = X1.shape; p = k - 1
    inv = np.linalg.inv(X1.T @ X1)                    # (X′X)⁻¹
    b = inv @ X1.T @ y                                # b = (X′X)⁻¹X′y
    yhat = X1 @ b; e = y - yhat
    sse, sst = e @ e, ((y - y.mean()) ** 2).sum(); ssr = sst - sse
    dfe = n - p - 1; mse = sse / dfe; s = math.sqrt(mse)
    se = np.sqrt(mse * np.diag(inv)); t = b / se
    h = np.einsum("ij,jk,ik->i", X1, inv, X1)         # leverage: x_i′(X′X)⁻¹x_i
    r2 = ssr / sst; F = (ssr / p) / mse
    return SimpleNamespace(b=b, se=se, t=t, p=2 * stats.t.sf(np.abs(t), dfe), yhat=yhat, e=e, sse=sse, sst=sst,
                           ssr=ssr, dfe=dfe, s=s, r2=r2, adj=1 - (1 - r2) * (n - 1) / dfe, F=F,
                           pF=stats.f.sf(F, p, dfe), inv=inv, h=h, std=e / (s * np.sqrt(1 - h)), n=n, k=p)

def logit(y, X, steps=60):
    """Logistic regression by maximum likelihood (Newton's method): b, standard errors, z and p (Wald), and the G test
    of the whole model, G = 2(LL − LL₀), with its chi-square p-value."""
    y = np.asarray(y, float); X1 = np.column_stack([np.ones(len(y)), np.asarray(X, float)])
    b = np.zeros(X1.shape[1])
    for _ in range(steps):
        pr = 1 / (1 + np.exp(-X1 @ b))
        H = X1.T @ (X1 * (pr * (1 - pr))[:, None])
        step = np.linalg.solve(H, X1.T @ (y - pr)); b = b + step
        if np.abs(step).max() < 1e-12: break
    pr = 1 / (1 + np.exp(-X1 @ b)); H = X1.T @ (X1 * (pr * (1 - pr))[:, None])
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    ll = (y * np.log(pr) + (1 - y) * np.log(1 - pr)).sum(); yb = y.mean()
    ll0 = len(y) * (yb * math.log(yb) + (1 - yb) * math.log(1 - yb)); G = 2 * (ll - ll0)
    return SimpleNamespace(b=b, se=se, z=b / se, pz=2 * stats.norm.sf(np.abs(b / se)), G=G,
                           pG=stats.chi2.sf(G, X1.shape[1] - 1), prob=pr)

def durbin_watson(e):
    """d = Σ(eₜ − eₜ₋₁)² / Σeₜ²: a spreadsheet's SUMXMY2(…)/SUMSQ(…)."""
    e = np.asarray(e, float); return np.sum(np.diff(e) ** 2) / np.sum(e ** 2)

def prob(eta): return 1 / (1 + np.exp(-eta))

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, sqrt(…), exp(…) and ln(…)."""
    fns = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log}
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in fns and len(n.args) == 1:
            return fns[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(km/h|kmh|minutes?|years?|dollars?|thousands?)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'The multiple regression model',
     "q": 'A regression gives ŷ = 22 + 3.5 x1 − 1.2 x2. Predict y when x1 = 10 and x2 = 5.',
     "parts": ['ŷ'],
     "answer": [51], "tol": [0.005], "function": 'SUMPRODUCT',
     "working": '22 + 35 − 6 = 51'},
    {"id": 'P1.2', "tier": 1, "group": 'The multiple regression model',
     "q": 'LINEST(sales, B2:C30, TRUE, TRUE) returns 2.4, −0.6, 15 in its first row, where column B is price and column C is advertising. What is the coefficient of price?',
     "parts": ['the coefficient of price'],
     "answer": [-0.6], "tol": [0.0005], "function": 'LINEST, INDEX',
     "working": 'LINEST lists the coefficients in reverse order of the x columns: advertising (2.4), then price (−0.6), then b0 (15)'},
    {"id": 'P1.3', "tier": 1, "group": 'Fit and significance',
     "q": 'A regression has SSR = 840 and SST = 1,000. What is R²?',
     "parts": ['R²'],
     "answer": [0.84], "tol": [0.0005], "function": '—',
     "working": 'R² = SSR/SST = 0.84'},
    {"id": 'P1.4', "tier": 1, "group": 'Fit and significance',
     "q": 'R² = 0.84 with n = 25 observations and p = 3 independent variables. What is the adjusted R²?',
     "parts": ['adjusted R²'],
     "answer": [0.8171428571428572], "tol": [0.0005], "function": '—',
     "working": '1 − (1 − 0.84) × 24/21 = 1 − 0.16 × 1.1429 ≈ 0.817'},
    {"id": 'P1.5', "tier": 1, "group": 'Fit and significance',
     "q": 'A coefficient is b2 = 4.2 with standard error 1.5. What is its t statistic?',
     "parts": ['t'],
     "answer": [2.8000000000000003], "tol": [0.0005], "function": '—',
     "working": 't = b2/s_b2 = 2.8'},
    {"id": 'P1.6', "tier": 1, "group": 'Prediction, categories and residuals',
     "q": 'Sales regions are North, South, East and West. How many dummy variables does a regression need for region?',
     "parts": ['dummy variables'],
     "answer": [3], "tol": [0], "function": '—',
     "working": 'k − 1 = 3 dummies; the region without one is the baseline'},
    {"id": 'P1.7', "tier": 1, "group": 'Logistic regression',
     "q": "A borrower's probability of default is 0.2. What are the odds of default?",
     "parts": ['odds of default'],
     "answer": [0.25], "tol": [0.0005], "function": '—',
     "working": 'odds = p/(1 − p) = 0.2/0.8 = 0.25, one to four'},
    {"id": 'P1.8', "tier": 1, "group": 'The general linear model',
     "q": 'ŷ = 10 + 2x + 3d + 1.5xd, where d is 1 or 0. What is the slope of ŷ against x when d = 1?',
     "parts": ['slope against x when d = 1'],
     "answer": [3.5], "tol": [0.0005], "function": '—',
     "working": 'slope = 2 + 1.5 × 1 = 3.5 (when d = 0 it is 2)'},
    {"id": 'P1.9', "tier": 1, "group": 'Autocorrelation',
     "q": 'Residuals in time order give Σ(e_t − e_t−1)² = 36 and Σe_t² = 20. What is the Durbin–Watson statistic?',
     "parts": ['Durbin–Watson d'],
     "answer": [1.8], "tol": [0.0005], "function": 'SUMXMY2, SUMSQ',
     "working": 'd = 36/20 = 1.8, close to 2'},
    {"id": 'P2.1', "tier": 2, "group": 'The multiple regression model',
     "q": 'On the forty flats (Table tab), fit rent on size and distance with LINEST. Give b0, b1 (size) and b2 (distance).',
     "parts": ['b₀', 'b₁ (size, $ per m²)', 'b₂ (distance, $ per km)'],
     "answer": [509.733372096779, 16.945455925676725, -40.490642300002946], "tol": [0.05, 0.005, 0.005], "function": 'LINEST, INDEX',
     "working": '=LINEST(rent, size:distance, TRUE, TRUE): ŷ = 509.73 + 16.945 size − 40.491 distance'},
    {"id": 'P2.2', "tier": 2, "group": 'Fit and significance',
     "q": "A model of 30 plants' monthly output on labour hours and machine hours has SST = 1,250 and SSE = 225. Give R², the adjusted R², F and its p-value.",
     "parts": ['R²', 'adjusted R²', 'F', 'p-value'],
     "answer": [0.82, 0.8066666666666666, 61.49999999999999, 8.834435796555232e-11], "tol": [0.0005, 0.0005, 0.005, 1e-09], "function": 'F.DIST.RT',
     "working": 'SSR = 1,025; R² = 0.82; adjusted 1 − 0.18 × 29/27 ≈ 0.8067; MSR = 512.5, MSE = 225/27 ≈ 8.333; F = 61.5; F.DIST.RT(61.5, 2, 27) ≈ 7 × 10⁻¹¹'},
    {"id": 'P2.3', "tier": 2, "group": 'Fit and significance',
     "q": "In the flats' model (rent on size and distance), test β2 (distance) = 0 at α = 0.05: t and the p-value.",
     "parts": ['t', 'p-value (two-tailed)'],
     "answer": [-4.809019502508942, 2.5411153735444003e-05], "tol": [0.005, 5e-06], "function": 'LINEST, T.DIST.2T',
     "working": 't = −40.491/8.420 ≈ −4.81, df 37; T.DIST.2T(4.81, 37) ≈ 0.00003: reject; distance matters, holding size fixed'},
    {"id": 'P2.4', "tier": 2, "group": 'Prediction, categories and residuals',
     "q": 'A model with n = 28 and p = 2 predicts ŷ = 84.0 at some x values, with s = 6.0 and the standard error of ŷ (for the mean) 1.6. Give the 95% confidence interval for the mean of y and the 95% prediction interval for one y.',
     "parts": ['confidence interval: lower end', 'confidence interval: upper end', 'prediction interval: lower end', 'prediction interval: upper end'],
     "answer": [80.70473831559472, 87.29526168440528, 71.21094548535012, 96.78905451464988], "tol": [0.005, 0.005, 0.005, 0.005], "function": 'T.INV.2T, SQRT',
     "working": 't(0.025, 25) = 2.060; CI 84 ± 2.060 × 1.6 = 80.70 … 87.30; for one y the standard error is √(6² + 1.6²) ≈ 6.210, so PI 84 ± 12.79 = 71.21 … 96.79'},
    {"id": 'P2.5', "tier": 2, "group": 'Prediction, categories and residuals',
     "q": "A flat's residual is −$210 in a model with s = $98.8, p = 3 and n = 40; its leverage is h = 0.12. Give its standardized residual. Is it unusual? Is its leverage high (rule: h > 3(p + 1)/n)?",
     "parts": ['standardized residual', 'the leverage rule 3(p + 1)/n'],
     "answer": [-2.265797086775212, 0.3], "tol": [0.005, 0.0005], "function": 'SQRT',
     "working": '−210/(98.8 × √0.88) ≈ −2.27: beyond ±2, worth a look; 3 × 4/40 = 0.30, and 0.12 is below it: not high leverage'},
    {"id": 'P2.6', "tier": 2, "group": 'Logistic regression',
     "q": "A subscription's logistic model: ln(odds of buying) = −3.2 + 0.04 × minutes on the trial. Give the probability of buying after 50 minutes, the odds ratio for 10 more minutes, and the minutes at which the probability is 0.5.",
     "parts": ['probability after 50 minutes', 'odds ratio for 10 more minutes', 'minutes where p = 0.5'],
     "answer": [0.23147521650098232, 1.4918246976412703, 80], "tol": [0.0005, 0.0005, 0.05], "function": 'EXP',
     "working": '−3.2 + 2 = −1.2; p = EXP(−1.2)/(1 + EXP(−1.2)) ≈ 0.231; odds ratio EXP(0.4) ≈ 1.492; p = 0.5 where the log odds are 0: 3.2/0.04 = 80 minutes'},
    {"id": 'P2.7', "tier": 2, "group": 'The general linear model',
     "q": 'On the 24 van routes, fit fuel on speed and speed squared. Give b1, b2 and the speed with the lowest predicted fuel use.',
     "parts": ['b₁ (speed)', 'b₂ (speed²)', 'the speed with the lowest fuel use (km/h)'],
     "answer": [-0.4903249606834237, 0.004283831168949925, 57.229725139192944], "tol": [0.0005, 5e-05, 0.05], "function": 'LINEST',
     "working": '=LINEST(fuel, speed:speed², TRUE, TRUE): ŷ = 26.84 − 0.4903 s + 0.004284 s²; lowest at 0.4903/(2 × 0.004284) ≈ 57.2 km/h'},
    {"id": 'P2.8', "tier": 2, "group": 'Building the model',
     "q": "A model with 5 x's on n = 40 has SSE = 700. Dropping two of them gives SSE = 820. Test whether the two add anything: the partial F and its p-value.",
     "parts": ['partial F', 'p-value'],
     "answer": [2.914285714285714, 0.06789396774436474], "tol": [0.005, 0.0005], "function": 'F.DIST.RT',
     "working": 'MSE(full) = 700/34 ≈ 20.59; F = (120/2)/20.59 ≈ 2.914; F.DIST.RT(2.914, 2, 34) ≈ 0.068: not significant at 0.05, keep the smaller model'},
    {"id": 'P2.9', "tier": 2, "group": 'Autocorrelation',
     "q": "The supplier's 24 months (Table tab): fit sales on housing starts, then compute the Durbin–Watson statistic of the residuals. With dL = 1.273 and dU = 1.446 (n = 24, one x, α = 0.05), what do you conclude?",
     "parts": ['d'],
     "answer": [0.8280094600905945], "tol": [0.0005], "function": 'LINEST, SUMXMY2, SUMSQ',
     "working": 'residuals from LINEST; d = SUMXMY2(e2:e24, e1:e23)/SUMSQ(e1:e24) ≈ 0.828 < 1.273: positive autocorrelation'},
    {"id": 'P3.1', "tier": 3, "group": 'The multiple regression model',
     "q": 'The flats: regress rent on distance alone, then on size and distance. Give both distance coefficients and the correlation of size with distance. Why does the sign flip?',
     "parts": ['distance, alone ($ per km)', 'distance, with size ($ per km)', 'r(size, distance)'],
     "answer": [56.98123726402412, -40.490642300002946, 0.7956544763135582], "tol": [0.005, 0.005, 0.0005], "function": 'LINEST, CORREL',
     "working": "alone +56.98 a km; with size −40.49 a km; r(size, distance) ≈ 0.796. Farther flats are bigger, so distance alone carries size's effect; holding size fixed, distance lowers rent"},
    {"id": 'P3.2', "tier": 3, "group": 'Fit and significance',
     "q": "Add rooms to the flats' model (rent on size, distance, rooms). Give the t statistic of rooms, the VIF of size, and the standard error of size's coefficient before and after. Is rooms useless?",
     "parts": ['t of rooms', 'VIF of size', 's_b of size, without rooms', 's_b of size, with rooms'],
     "answer": [-0.1054090337709832, 8.02257480855545, 1.1646530159375392, 2.025489496825357], "tol": [0.005, 0.005, 0.0005, 0.0005], "function": 'LINEST, CORREL',
     "working": 't(rooms) ≈ −0.11 (p ≈ 0.92); VIF(size) ≈ 8.02; s_b1 1.165 → 2.025. Not useless: rooms alone (with distance) has t ≈ 6.8. It says almost the same thing as size (r ≈ 0.90), so it adds nothing once size is in'},
    {"id": 'P3.3', "tier": 3, "group": 'Fit and significance',
     "q": "Compare R² and the adjusted R² of the flats' model with and without rooms. Which measure should you use to compare them, and why?",
     "parts": ['R², without rooms', 'R², with rooms', 'adjusted R², without rooms', 'adjusted R², with rooms'],
     "answer": [0.9009410730381451, 0.9009716372191545, 0.8955865364456124, 0.8927192736540841], "tol": [5e-05, 5e-05, 5e-05, 5e-05], "function": 'LINEST, INDEX',
     "working": "R² 0.90094 → 0.90097 (it can't fall when a variable is added); adjusted 0.8956 → 0.8927 (it falls): compare models with different numbers of x's on the adjusted R²"},
    {"id": 'P3.4', "tier": 3, "group": 'Prediction, categories and residuals',
     "q": "Add parking (1/0) to the flats' model. Give its coefficient and 95% confidence interval. What does it mean, and what does the model assume about parking and distance?",
     "parts": ['b₃ (parking, $ a month)', '95% interval: lower end', '95% interval: upper end'],
     "answer": [75.1081889759721, 7.284366821416128, 142.93201113052805], "tol": [0.005, 0.05, 0.05], "function": 'LINEST, T.INV.2T',
     "working": 'b3 ≈ 75.11, s ≈ 33.44, t(0.025, 36) ≈ 2.028: 7.3 … 142.9 dollars. Parking adds about $75 a month, holding size and distance fixed; the model assumes the same $75 at every size and distance (parallel planes)'},
    {"id": 'P3.5', "tier": 3, "group": 'Logistic regression',
     "q": 'The sixty loans, with debt-to-income and guarantor: the fitted model is ln(odds) = b0 + b1 DTI + b2 guarantor (Python version, or the values given on the sheet). At DTI = 40, give the probability of default without and with a guarantor, and the odds ratio for a guarantor. Why is the odds ratio not the ratio of the probabilities?',
     "parts": ['p at DTI 40, no guarantor', 'p at DTI 40, with a guarantor', 'odds ratio for a guarantor'],
     "answer": [0.41377481504355296, 0.049951745489365075, 0.07449127766991191], "tol": [0.0005, 0.0005, 0.0005], "function": 'EXP',
     "working": "b = −4.180, 0.0959, −2.597; at 40: p ≈ 0.414 without, ≈ 0.050 with; odds ratio EXP(−2.597) ≈ 0.074. The odds ratio multiplies the odds, p/(1 − p); the probabilities' ratio is 0.050/0.414 ≈ 0.12 and changes with DTI"},
    {"id": 'P3.6', "tier": 3, "group": 'The general linear model',
     "q": 'The 30 forklifts: fit LN(cost) on age. Give the yearly percent rise in cost, and the predicted cost of a 10-year-old forklift. Then fit cost on age directly: what do its residuals do as ŷ grows?',
     "parts": ['yearly rise (a decimal: 0.117 for 11.7%)', 'predicted cost at 10 years ($)'],
     "answer": [0.11712607812852593, 2765.402015098544], "tol": [0.0005, 1], "function": 'LN, EXP, LINEST',
     "working": "ln ŷ = 6.817 + 0.1108 age: EXP(0.1108) − 1 ≈ 11.7% a year; at 10 years EXP(6.817 + 1.108) ≈ $2,765. The straight-line fit's residuals fan out as ŷ grows, so its intervals are wrong at both ends"},
    {"id": 'P3.7', "tier": 3, "group": 'Building the model',
     "q": "Forward selection tries 20 candidate x's that are pure noise, each tested at α = 0.05. If the tests were independent, what is the chance that at least one enters? With 40 candidates?",
     "parts": ['20 candidates', '40 candidates'],
     "answer": [0.6415140775914581, 0.8714878434348969], "tol": [0.0005, 0.0005], "function": '—',
     "working": '1 − 0.95²⁰ ≈ 0.642; 1 − 0.95⁴⁰ ≈ 0.871. A variable that survives a search has a p-value that is too small'},
    {"id": 'P3.8', "tier": 3, "group": 'Building the model',
     "q": 'Three packing-line layouts, five runs each (units an hour): first 52, 55, 49, 54, 50; second 58, 61, 57, 60, 59; third 54, 51, 56, 53, 56. Fit a regression on two dummies (second, third). Give b0, b1, b2 and F. What test is this?',
     "parts": ['b₀', 'b₁ (second)', 'b₂ (third)', 'F'],
     "answer": [52.00000000000001, 6.999999999999993, 1.9999999999999858, 14.444444444444445], "tol": [0.005, 0.005, 0.005, 0.005], "function": 'LINEST, F.DIST.RT',
     "working": "b0 = 52 (the first layout's mean), b1 = 7, b2 = 2 (differences from it); F = (130/2)/(54/12) ≈ 14.44, p ≈ 0.0006: the one-way ANOVA F test, run as a regression"},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": 'Reproduce the tempting comparison: the mean price of good and of fair vans, the gap, and the mean age of each group. Why does the gap overstate what condition is worth?',
     "parts": ['mean price, good ($)', 'mean price, fair ($)', 'the gap ($)'],
     "answer": [26154.772727272728, 18714.934210526317, 7439.838516746411], "tol": [0.5, 0.5, 0.5], "function": 'AVERAGEIFS',
     "working": 'AVERAGEIFS: good $26,155, fair $18,715, gap $7,440; good vans average 3.35 years, fair 5.01: younger vans are in better condition, so the gap mixes age into condition'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": 'Fit LN(price) on age, thousand km, hybrid, electric and good (dummies 1/0; diesel and fair are the baselines). Give the five slopes and R².',
     "parts": ['age', 'thousand km', 'hybrid', 'electric', 'good', 'R²'],
     "answer": [-0.10360704251408359, -0.0014981996331915673, 0.05153189666304392, 0.11658605297642377, 0.10157104781127302, 0.9122776746590543], "tol": [5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05], "function": 'LN, LINEST',
     "working": '=LINEST(LN(price), age:good, TRUE, TRUE): −0.10361, −0.00150, 0.05153, 0.11659, 0.10157; R² ≈ 0.912'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": 'Turn the good-condition coefficient into a percent premium, with its 95% interval.',
     "parts": ['premium (a decimal: 0.107 for 10.7%)', '95% interval: lower end', '95% interval: upper end'],
     "answer": [0.10690855902867846, 0.07818101813265432, 0.13640152946951334], "tol": [0.0005, 0.0005, 0.0005], "function": 'EXP, T.INV.2T',
     "working": 'EXP(0.10157) − 1 ≈ 10.7%; b ± t s_b with t(0.025, 234) ≈ 1.970 and s_b ≈ 0.01335, then EXP − 1: 7.8% … 13.6%'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": 'Predict the price of a three-year-old diesel van with 80,000 km in fair condition, with a 95% prediction interval in dollars.',
     "parts": ['predicted price ($)', '95% prediction interval: lower end ($)', '95% prediction interval: upper end ($)'],
     "answer": [23183.30957965579, 19425.09113014104, 27668.639465592947], "tol": [1, 1, 1], "function": 'TREND, EXP',
     "working": 'ln ŷ = 10.48186 − 0.10361 × 3 − 0.00150 × 80 ≈ 10.0512; EXP → $23,183; interval $19,425 … $27,669 (exponentiate the ends of the interval for ln price)'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": 'Does fuel type matter? Test the two fuel dummies together with a partial F test.',
     "parts": ['partial F', 'p-value'],
     "answer": [30.192872808481642, 2.1626805402087044e-12], "tol": [0.05, 1e-09], "function": 'LINEST, F.DIST.RT',
     "working": 'SSE without the two dummies vs with them; F ≈ 30.2 on 2 and 234 df; p ≈ 2e-12: fuel type matters'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": "Age and km are correlated. Give their correlation, the VIF of each, and km's coefficient when age is left out. Should you drop km?",
     "parts": ['r(age, km)', 'VIF of age', 'VIF of km', "km's coefficient without age"],
     "answer": [0.905093559264762, 5.9293837197401205, 5.70533625181387, -0.004645074120526893], "tol": [0.0005, 0.005, 0.005, 5e-05], "function": 'CORREL, LINEST',
     "working": "r ≈ 0.905; VIF 5.93 and 5.71, under 10; without age km's coefficient becomes -0.00465, about three times as steep, because it then carries age too. Keep both: each is significant and the predictions need both"},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": 'A second question: which vans sell at their first auction? Fit a logistic regression of sold_first on age and good (Python version). Give the odds ratio per year of age, the odds ratio for good condition, and the probability for a three-year-old good van.',
     "parts": ['odds ratio per year of age', 'odds ratio for good condition', 'p, a three-year-old good van'],
     "answer": [0.6788386057380944, 1.9162616891166773, 0.8830369266476964], "tol": [0.0005, 0.0005, 0.0005], "function": 'EXP',
     "working": 'b = 2.533, -0.387, 0.650; each year multiplies the odds by 0.679; good condition by 1.92 (z ≈ 1.73, p ≈ 0.08: not clear at 0.05); a three-year-old good van ≈ 0.883'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Corrowby Fleet Leasing',
     "q": "Write the recommendation. Anchor it: the predicted fair-condition price above which a $1,500 refurbishment pays at the premium's low end, and how many of the 152 fair vans clear it.",
     "parts": ["the break-even fair price at the premium's low end ($)", 'fair vans predicted above it'],
     "answer": [19186.242848038408, 59], "tol": [1, 0], "function": 'EXP, COUNTIFS',
     "working": "low-end premium 7.82%; 1,500/0.0782 ≈ $19,186; 59 of 152 fair vans are predicted above it. Refurbish those; for vans between $14,031 and $19,186 it probably pays but isn't sure; run a refurbishment trial, because the premium comes from vans that were good, not vans that were refurbished"},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 34 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [9, 9, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't, arithmetic typed either way
assert right(read_part("22+3.5*10-1.2*5"), 51, 0.005) and not right(read_part("50"), 51, 0.005)
assert right(read_part("exp(-1.2)/(1+exp(-1.2))"), Q["P2.6"]["answer"][0], 0.0005)
print("statsmodels:", "ready" if sm is not None else "not installed here; the numpy versions below do the same fits")

## The data sets

Every data set the course teaches with, as pandas DataFrames: the same numbers as the spreadsheet's Table tab.

| DataFrame | rows | what it is |
|---|---|---|
| `flats` | 40 | one city's flats: size (m²), distance to the centre (km), rooms, parking (1/0), building age, floor, balcony (1/0), monthly rent ($) |
| `loans` | 60 | personal loans: debt-to-income ratio (%), guarantor (1/0), default within two years (1/0) |
| `routes` | 24 | delivery-van routes: average speed (km/h), fuel use (litres per 100 km) |
| `warehouses` | 32 | warehouse-months: heating degree days, insulated (1/0), heating cost ($) |
| `forklifts` | 30 | forklifts: age (years), last year's maintenance cost ($) |
| `supplier` | 24 | a building supplier's months: regional housing starts, sales ($ thousands) |
| `layouts` | 15 | three packing-line layouts, five runs each: units an hour |

In [ ]:
# @DATA
# the forty flats: size (m²), distance (km), rooms, parking (1/0), building age, floor, balcony (1/0), rent ($)
FLATS = {
    'size': [55, 64, 38, 107, 66, 40, 75, 62, 42, 85, 85, 102, 103, 87, 104, 100, 80, 70, 73, 59, 52, 64, 42, 104,
             118, 76, 42, 117, 93, 89, 55, 106, 59, 52, 108, 113, 72, 75, 68, 93],
    'distance': [0.6, 3.8, 0.6, 8.3, 4.1, 0.6, 6.1, 5.2, 0.6, 3.2, 5.2, 8.5, 5.4, 3.7, 8.1, 9.9, 4.7, 1.1, 4.5, 5.2,
                 4.3, 1.9, 0.8, 13.8, 6.2, 3.3, 0.6, 6.5, 4.5, 6.5, 1.7, 11.9, 4.3, 4.1, 8.5, 9.6, 3.1, 5.1, 4.5,
                 10.2],
    'rooms': [2, 3, 1, 5, 2, 1, 3, 2, 2, 4, 3, 4, 4, 3, 4, 4, 3, 3, 3, 2, 2, 3, 1, 4, 5, 3, 2, 4, 3, 3, 2, 3, 3, 2,
              4, 4, 3, 3, 2, 4],
    'parking': [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0,
                0, 0, 0, 1, 1, 0],
    'age': [9, 29, 39, 29, 24, 8, 47, 56, 44, 12, 47, 60, 60, 49, 32, 36, 13, 57, 24, 46, 14, 16, 45, 45, 15, 2,
            31, 15, 26, 22, 55, 48, 23, 47, 26, 16, 9, 9, 30, 60],
    'floor': [7, 9, 1, 11, 3, 2, 8, 8, 2, 8, 9, 8, 4, 8, 5, 9, 8, 0, 4, 5, 4, 0, 6, 9, 0, 8, 2, 9, 4, 12, 6, 0, 9,
              7, 12, 9, 7, 12, 2, 4],
    'balcony': [0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1,
                1, 1, 0, 1, 0, 0],
    'rent': [1345, 1395, 1085, 1955, 1420, 1220, 1295, 1425, 1270, 1950, 1730, 1960, 1955, 1590, 1940, 1745, 1845,
             1745, 1580, 1195, 1255, 1435, 1175, 1795, 2380, 1590, 1260, 2285, 2010, 1785, 1185, 1900, 1365, 1370,
             1990, 1890, 1575, 1625, 1630, 1560],
}
# the sixty loans: debt-to-income (%), guarantor (1/0), default within two years (1/0)
LOANS = {
    'dti': [45, 20, 30, 24, 34, 8, 12, 50, 50, 19, 47, 51, 36, 22, 51, 50, 50, 16, 20, 47, 13, 52, 39, 12, 10,
            44, 39, 25, 46, 30, 16, 50, 55, 17, 44, 17, 31, 39, 11, 54, 58, 15, 21, 15, 32, 52, 48, 37, 58, 18,
            15, 47, 20, 31, 14, 34, 55, 27, 33, 28],
    'guarantor': [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1,
                  1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
    'default': [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
                1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0],
}
# the 24 routes: average speed (km/h), fuel (litres per 100 km)
ROUTES = {
    'speed': [29, 43, 61, 26, 42, 87, 69, 56, 26, 62, 70, 43, 93, 44, 63, 68, 87, 79, 75, 68, 67, 79, 42, 81],
    'fuel': [16.1, 13.8, 13.0, 16.6, 14.6, 16.7, 12.9, 12.7, 16.6, 12.6, 12.4, 14.1, 18.5, 13.8, 13.5, 14.3, 16.5,
             14.6, 14.2, 12.0, 12.5, 16.1, 14.2, 15.2],
}
# the 32 warehouse-months: heating degree days, insulated (1/0), heating cost ($)
WAREHOUSES = {
    'hdd': [468, 503, 632, 421, 164, 277, 482, 340, 390, 202, 443, 358, 293, 349, 231, 466, 466, 647, 596, 641,
            234, 642, 251, 254, 243, 644, 644, 334, 362, 183, 319, 151],
    'insulated': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    'cost': [6000, 4610, 7690, 3610, 3520, 2770, 6280, 3530, 5480, 2550, 5790, 4000, 4650, 4010, 3890, 4070, 5860,
             5190, 7610, 5130, 4060, 4960, 3680, 2930, 4360, 4980, 7770, 3320, 5130, 2760, 4560, 2800],
}
# the 30 forklifts: age (years), last year's maintenance cost ($)
FORKLIFTS = {
    'age': [13.4, 2.1, 11.9, 5.0, 8.9, 3.1, 6.6, 13.9, 13.3, 4.5, 1.5, 5.1, 5.9, 8.3, 4.1, 2.8, 7.6, 3.4, 12.6,
            12.8, 10.9, 13.8, 13.5, 10.3, 7.0, 13.0, 7.5, 12.5, 12.7, 13.0],
    'cost': [4510, 1120, 3440, 1150, 2000, 1070, 1460, 5540, 3300, 1410, 1360, 1690, 2190, 1970, 1750, 1280, 3000,
             1330, 4250, 2780, 3320, 3610, 5170, 4020, 1570, 3820, 2360, 3170, 2940, 4300],
}
# the supplier's 24 months: housing starts, sales ($ thousands)
SUPPLIER = {
    'starts': [1874, 1575, 1658, 1470, 1376, 1478, 1427, 1496, 1787, 1489, 1657, 1495, 1398, 1454, 1671, 1641, 1110,
               1658, 1940, 1694, 1763, 1479, 1856, 1782],
    'sales': [781.1, 698.3, 684.8, 617.7, 579.5, 621.3, 634.1, 567.5, 678.5, 620.9, 691.5, 658.9, 679.0, 683.0,
              753.4, 754.3, 575.5, 725.3, 781.9, 651.7, 646.5, 659.2, 777.2, 711.8],
}
# three packing-line layouts, five runs each: units an hour
LAYOUTS = {'first': [52, 55, 49, 54, 50], 'second': [58, 61, 57, 60, 59], 'third': [54, 51, 56, 53, 56]}
# @END DATA
flats = pd.DataFrame(FLATS); loans = pd.DataFrame(LOANS); routes = pd.DataFrame(ROUTES)
warehouses = pd.DataFrame(WAREHOUSES); forklifts = pd.DataFrame(FORKLIFTS); supplier = pd.DataFrame(SUPPLIER)
layouts = pd.DataFrame({"layout": [k for k in LAYOUTS for _ in LAYOUTS[k]], "units": [v for k in LAYOUTS for v in LAYOUTS[k]]})
assert len(flats) == 40 and flats["parking"].sum() == 18 and len(loans) == 60 and loans["default"].sum() == 14
print("flats", flats.shape, " loans", loans.shape, " routes", routes.shape, " warehouses", warehouses.shape,
      " forklifts", forklifts.shape, " supplier", supplier.shape, " layouts", layouts.shape)
flats.head()

## Tier 1 · Warm-up

One idea, one step, small numbers: a prediction from the equation, reading LINEST's first row, R², the adjusted R²,
a t statistic, how many dummies, odds from a probability, a slope with an interaction, and the Durbin–Watson d.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

### How you'd do this in Python

The warm-ups are one line of arithmetic each. Note the one trap: a spreadsheet's LINEST lists the coefficients **back
to front** (the last x column first, the intercept last); Python's fits list them in the order you gave the columns,
intercept first. Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
yhat = 22 + 3.5 * 10 - 1.2 * 5                                   # P1.1
price = [2.4, -0.6, 15][1]                                       # P1.2: LINEST's row: advertising, price, then b0
r2 = 840 / 1000                                                  # P1.3: R² = SSR/SST
adj = 1 - (1 - 0.84) * (25 - 1) / (25 - 3 - 1)                  # P1.4: 1 − (1 − R²)(n − 1)/(n − p − 1)
t = 4.2 / 1.5                                                    # P1.5: t = b/s_b
dummies = 4 - 1                                                  # P1.6: k − 1; the fourth region is the baseline
odds = 0.2 / (1 - 0.2)                                           # P1.7: odds = p/(1 − p)
slope_d1 = 2 + 1.5 * 1                                           # P1.8: b1 + b3 d
d = 36 / 20                                                      # P1.9: Durbin–Watson
print(f"ŷ {yhat}   price's coefficient {price}   R² {r2}   adjusted {adj:.4f}   t {t:.2f}   dummies {dummies}   "
      f"odds {odds}   slope {slope_d1}   d {d}")
for k, v in (("P1.1", [yhat]), ("P1.2", [price]), ("P1.3", [r2]), ("P1.4", [adj]), ("P1.5", [t]), ("P1.6", [dummies]),
             ("P1.7", [odds]), ("P1.8", [slope_d1]), ("P1.9", [d])):
    assert agrees(k, v), k

### The forty flats, in a few lines

Rent on size and distance. `np.linalg.lstsq` gives the least-squares b's; `ols` (defined in the first cell) gives
everything a spreadsheet's LINEST gives, in the usual order: b₀ = 509.73, $16.95 a square metre holding distance
fixed, and **−$40.49 a kilometre holding size fixed**. On its own, distance seemed to *raise* the rent by $56.98 a
kilometre: farther flats tend to be bigger (r ≈ 0.80), so distance alone was carrying size's effect.

In [ ]:
X2 = flats[["size", "distance"]]
b, *_ = np.linalg.lstsq(np.column_stack([np.ones(len(flats)), X2]), flats["rent"], rcond=None)
fit2 = ols(flats["rent"], X2)
coefs = pd.DataFrame({"b": fit2.b, "standard error": fit2.se, "t": fit2.t, "p-value": fit2.p},
                     index=["intercept", "size", "distance"])
print(coefs.round(6).to_string())
print(f"\nR² {fit2.r2:.4f}   adjusted R² {fit2.adj:.4f}   s {fit2.s:.2f}   F {fit2.F:.2f} on 2 and {fit2.dfe} df, "
      f"p {fit2.pF:.1e}")
alone = ols(flats["rent"], flats[["distance"]])
print(f"distance alone: {alone.b[1]:+.2f} a km;  holding size fixed: {fit2.b[2]:+.2f} a km;  "
      f"r(size, distance) = {flats['size'].corr(flats['distance']):.3f}")
assert np.allclose(b, fit2.b) and [round(v, 2) for v in fit2.b] == [509.73, 16.95, -40.49]          # the video's numbers
assert [round(v, 2) for v in fit2.se] == [64.16, 1.16, 8.42] and round(fit2.t[2], 2) == -4.81 and round(fit2.F, 2) == 168.26
assert round(alone.b[1], 2) == 56.98 and round(fit2.r2, 2) == 0.90

In [ ]:
# The same fit with statsmodels, which prints the whole table (where statsmodels is installed).
if sm is not None:
    sm_fit = sm.OLS(flats["rent"], sm.add_constant(X2)).fit()
    print(sm_fit.params.round(4).to_string(), "\n\nstandard errors\n" + sm_fit.bse.round(4).to_string())
    assert np.allclose(sm_fit.params, fit2.b) and np.allclose(sm_fit.bse, fit2.se) and abs(sm_fit.fvalue - fit2.F) < 1e-6
else:
    print("statsmodels isn't installed here; the ols() fit above is the same.")

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
xs = np.linspace(0, 15, 50)
ax.plot(xs, alone.b[0] + alone.b[1] * xs, color=CORAL, ls="--", lw=2, label=f"distance alone: {alone.b[1]:+.2f} a km")
for lo, hi, col in ((35, 65, "#4f8f94"), (65, 90, SKY), (90, 120, "#a9dde0")):     # three size bands
    band = flats[(flats["size"] >= lo) & (flats["size"] < hi)]
    ax.scatter(band["distance"], band["rent"], s=55, color=col, zorder=3, label=f"{lo}–{hi} m²")
    mid = band["size"].mean()
    ax.plot(xs, fit2.b[0] + fit2.b[1] * mid + fit2.b[2] * xs, color=col, lw=2)
ax.set_xlim(0, 15); ax.set_xlabel("distance to the centre (km)"); ax.set_ylabel("rent ($ a month)")
ax.legend(loc="upper left", frameon=False, fontsize=11, ncol=2)
fig.suptitle(f"Holding size fixed, each km lowers the rent by ${-fit2.b[2]:.2f}", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig); plt.show()

### Slide a flat's size and distance

The fitted plane predicts the rent of a flat with a given size and distance. Two intervals sit around it. The narrow
**confidence interval** (SKY) is for the *mean* rent of all such flats; the wide **prediction interval** (VIOLET) is
for *one* flat. At 70 m² and 5 km: ŷ ≈ $1,493.46, the confidence interval about $1,456 to $1,531, the prediction
interval about $1,279 to $1,708, nearly six times as wide. Both come from x₀′(X′X)⁻¹x₀: s_ŷ = s√(x₀′(X′X)⁻¹x₀), and
for one flat s√(1 + x₀′(X′X)⁻¹x₀).

In [ ]:
def intervals(fit, x0, conf=0.95):
    """ŷ at x0 (the x values, without the 1), the confidence interval for the mean and the prediction interval."""
    v = np.r_[1.0, x0]; yh = v @ fit.b; q = v @ fit.inv @ v
    t = stats.t.ppf(1 - (1 - conf) / 2, fit.dfe)
    s_mean, s_ind = fit.s * math.sqrt(q), fit.s * math.sqrt(1 + q)
    return yh, (yh - t * s_mean, yh + t * s_mean), (yh - t * s_ind, yh + t * s_ind), s_mean, s_ind

def flat_picture(size=70.0, distance=5.0):
    yh, (c0, c1), (p0, p1), sm_, si_ = intervals(fit2, [size, distance])
    fig, ax = plt.subplots(figsize=(11.5, 3.4))
    fig.subplots_adjust(left=0.06, right=0.98, top=0.78, bottom=0.2)
    ax.plot([p0, p1], [0.35, 0.35], color=VIOLET, lw=10, solid_capstyle="butt", label="one flat (prediction)")
    ax.plot([c0, c1], [0.7, 0.7], color=SKY, lw=10, solid_capstyle="butt", label="mean of all such flats (confidence)")
    ax.scatter([yh], [0.525], s=120, color=GOLD, zorder=4)
    ax.set_ylim(0, 1); ax.set_yticks([]); ax.set_xlim(yh - 400, yh + 400); ax.set_xlabel("rent ($ a month)")
    ax.legend(loc="upper left", frameon=False, ncol=2, fontsize=11)
    fig.suptitle(f"{size:g} m², {distance:g} km: ŷ = ${yh:,.2f}; mean ${c0:,.0f} to ${c1:,.0f}; one flat ${p0:,.0f} to "
                 f"${p1:,.0f}", x=0.06, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

_yh, _ci, _pi, _sm, _si = intervals(fit2, [70, 5])                                     # the video's numbers
assert round(_yh, 2) == 1493.46 and round(_sm, 2) == 18.50 and round(_si, 2) == 105.67
assert [round(v) for v in _ci] == [1456, 1531] and [round(v) for v in _pi] == [1279, 1708]

In [ ]:
interact(flat_picture, size=FloatSlider(value=70, min=38, max=118, step=1, description="size (m²)"),
         distance=FloatSlider(value=5, min=0.5, max=14, step=0.1, description="distance (km)"));

## Tier 2 · Core

The module's standard problems, several steps each: the flats' plane, R², the adjusted R² and F from the sums of
squares, a t test with the other x held fixed, the two intervals, a standardized residual and leverage, a logistic
model's probability and odds ratio, a squared term, a partial F test, and the Durbin–Watson test.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

### How you'd do this in Python

**The fit and its tests**: `ols(y, X)` returns the b's, their standard errors, t and p, R², the adjusted R², F and
its p-value. **The t and F distributions**: `stats.t.ppf(0.975, df)` is a spreadsheet's T.INV.2T(0.05, df), and
`stats.f.sf(F, d1, d2)` is F.DIST.RT(F, d1, d2). **A squared term** is one more column: `speed**2`. **A partial F
test** compares the SSE of two fits. **Durbin–Watson**: `np.diff` gives eₜ − eₜ₋₁.

In [ ]:
assert agrees("P2.1", list(fit2.b))                                                          # P2.1
n, p, sst, sse = 30, 2, 1250, 225                                                            # P2.2: thirty plants
r2, adj = (sst - sse) / sst, 1 - (sse / sst) * (n - 1) / (n - p - 1)
F = ((sst - sse) / p) / (sse / (n - p - 1))
assert agrees("P2.2", [r2, adj, F, stats.f.sf(F, p, n - p - 1)])
assert agrees("P2.3", [fit2.t[2], fit2.p[2]])                                                # P2.3: distance's t test
q = stats.t.ppf(0.975, 25); s_one = math.sqrt(6 ** 2 + 1.6 ** 2)                              # P2.4: the two intervals
assert agrees("P2.4", [84 - q * 1.6, 84 + q * 1.6, 84 - q * s_one, 84 + q * s_one])
assert agrees("P2.5", [-210 / (98.8 * math.sqrt(1 - 0.12)), 3 * (3 + 1) / 40])               # P2.5
assert agrees("P2.6", [prob(-3.2 + 0.04 * 50), math.exp(10 * 0.04), 3.2 / 0.04])             # P2.6: the trial
routes["speed2"] = routes["speed"] ** 2                                                      # P2.7: a squared term
fq = ols(routes["fuel"], routes[["speed", "speed2"]])
best = -fq.b[1] / (2 * fq.b[2])
print(f"routes: ŷ = {fq.b[0]:.2f} {fq.b[1]:+.4f} speed {fq.b[2]:+.6f} speed²; R² {fq.r2:.2f} (a straight line: "
      f"{ols(routes['fuel'], routes[['speed']]).r2:.3f}); lowest at {best:.1f} km/h")
assert agrees("P2.7", [fq.b[1], fq.b[2], best])
F = ((820 - 700) / 2) / (700 / 34)                                                           # P2.8: partial F
assert agrees("P2.8", [F, stats.f.sf(F, 2, 34)])
fs = ols(supplier["sales"], supplier[["starts"]])                                            # P2.9: Durbin–Watson
d = durbin_watson(fs.e)
print(f"supplier: d = {d:.3f}, below dL = 1.273: positive autocorrelation")
assert agrees("P2.9", [d])

### The loans' logistic regression, in three lines

Whether a loan defaults is 1 or 0, so a straight line won't do: it predicts probabilities below zero and above one.
Logistic regression models the log odds, ln(p/(1 − p)) = b₀ + b₁x, and its b's are found by **maximum likelihood**:
the values that make the observed defaults as probable as they can be. No single spreadsheet function fits it. With
statsmodels it's three lines; the `logit` function in the first cell does the same with numpy (Newton's method), so
both give b₀ ≈ −4.84 and b₁ ≈ 0.0952.

In [ ]:
if sm is not None:                                    # three lines with statsmodels
    X = sm.add_constant(loans[["dti"]])
    fit = sm.Logit(loans["default"], X).fit(disp=0)
    print(fit.params.to_string())
else:
    print("statsmodels isn't installed here; the numpy fit below gives the same b's.")

In [ ]:
lg1 = logit(loans["default"], loans[["dti"]])                                                 # the same, with numpy
print(f"b0 = {lg1.b[0]:.4f}, b1 = {lg1.b[1]:.5f}: each point of debt-to-income multiplies the odds of default by "
      f"{math.exp(lg1.b[1]):.3f}; ten points by {math.exp(10 * lg1.b[1]):.2f}")
print(f"the whole model: G = {lg1.G:.1f} on 1 df, p = {lg1.pG:.4f}")
if sm is not None: assert np.allclose(fit.params, lg1.b) and np.allclose(fit.bse, lg1.se)
assert round(lg1.b[0], 2) == -4.84 and round(lg1.b[1], 4) == 0.0952                          # the video's numbers
assert round(math.exp(10 * lg1.b[1]), 2) == 2.59 and round(lg1.G, 1) == 14.9
assert round(prob(lg1.b[0] + 40 * lg1.b[1]), 2) == 0.26 and round(prob(lg1.b[0] + 20 * lg1.b[1]), 2) == 0.05

### Slide a borrower along the S-curve

The probability of default against the debt-to-income ratio, without a guarantor (SKY) and with one (MINT). At 40 %
the model gives about 0.41 without a guarantor and 0.05 with one. The guarantor's odds ratio, e^b₂ ≈ 0.074, is the
same at every ratio; the ratio of the two probabilities isn't.

In [ ]:
lg2 = logit(loans["default"], loans[["dti", "guarantor"]])

def loan_picture(dti=40, guarantor=False):
    xs = np.linspace(0, 70, 200)
    p_no, p_g = prob(lg2.b[0] + lg2.b[1] * xs), prob(lg2.b[0] + lg2.b[1] * xs + lg2.b[2])
    here = prob(lg2.b[0] + lg2.b[1] * dti + lg2.b[2] * guarantor)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    jitter = loans["default"] + np.where(loans["guarantor"] == 1, -0.03, 0.03)
    ax.scatter(loans["dti"], jitter, s=30, color=np.where(loans["guarantor"] == 1, MINT, CHALK), alpha=0.8, zorder=3)
    ax.plot(xs, p_no, color=SKY, label="no guarantor"); ax.plot(xs, p_g, color=MINT, label="with a guarantor")
    ax.scatter([dti], [here], s=140, color=GOLD, zorder=5)
    ax.set_xlim(0, 70); ax.set_ylim(-0.08, 1.08); ax.set_xlabel("debt-to-income ratio (%)"); ax.set_ylabel("P(default)")
    ax.legend(loc="center right", frameon=False)
    fig.suptitle(f"DTI {dti}%{' with a guarantor' if guarantor else ''}: P(default) = {here:.3f}, odds {here / (1 - here):.3f}",
                 x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert round(prob(lg2.b[0] + 40 * lg2.b[1]), 3) == 0.414 and round(math.exp(lg2.b[2]), 3) == 0.074

In [ ]:
interact(loan_picture, dti=IntSlider(value=40, min=0, max=70, step=1, description="DTI (%)"),
         guarantor=Checkbox(value=False, description="guarantor"));

## Tier 3 · Stretch

Combine ideas, choose the method, spot the trap, work backwards: why distance's sign flips, a variable that looks
useless because it repeats another, R² against the adjusted R², what a dummy assumes, odds ratios against
probabilities, a log model, the danger of searching many x's, and a one-way ANOVA run as a regression.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**Multicollinearity**: the variance inflation factor of an x is 1/(1 − R²) of that x on the other x's; statsmodels'
`variance_inflation_factor` gives the same. **A dummy** is a column of 1s and 0s (`pd.get_dummies`), one fewer than
the categories. **A log model**: fit `np.log(cost)`, then e^b − 1 is the percent change. **A logistic model with two
x's** is the same `logit` call with two columns.

In [ ]:
r_sd = flats["size"].corr(flats["distance"])                                                 # P3.1: the sign flip
assert agrees("P3.1", [alone.b[1], fit2.b[2], r_sd])
f3 = ols(flats["rent"], flats[["size", "distance", "rooms"]])                                # P3.2: add rooms
vif_size = 1 / (1 - ols(flats["size"], flats[["distance", "rooms"]]).r2)
print(f"rooms: t = {f3.t[3]:.2f} (p {f3.p[3]:.2f}); VIF of size {vif_size:.2f}; size's standard error "
      f"{fit2.se[1]:.3f} → {f3.se[1]:.3f}")
assert agrees("P3.2", [f3.t[3], vif_size, fit2.se[1], f3.se[1]])
if sm is not None:
    Xr = sm.add_constant(flats[["size", "distance", "rooms"]]).to_numpy(float)
    assert abs(variance_inflation_factor(Xr, 1) - vif_size) < 1e-9
assert agrees("P3.3", [fit2.r2, f3.r2, fit2.adj, f3.adj])                                    # P3.3: adjusted R²
fp = ols(flats["rent"], flats[["size", "distance", "parking"]]); q = stats.t.ppf(0.975, fp.dfe)   # P3.4: a dummy
assert agrees("P3.4", [fp.b[3], fp.b[3] - q * fp.se[3], fp.b[3] + q * fp.se[3]])
p_no, p_g = prob(lg2.b[0] + 40 * lg2.b[1]), prob(lg2.b[0] + 40 * lg2.b[1] + lg2.b[2])         # P3.5
print(f"loans at DTI 40: p {p_no:.3f} without, {p_g:.3f} with a guarantor; their ratio {p_g / p_no:.2f}, "
      f"the odds ratio {math.exp(lg2.b[2]):.3f}")
assert agrees("P3.5", [p_no, p_g, math.exp(lg2.b[2])])
fl = ols(np.log(forklifts["cost"]), forklifts[["age"]])                                      # P3.6: a log model
print(f"forklifts: ln ŷ = {fl.b[0]:.3f} + {fl.b[1]:.4f} age: {math.exp(fl.b[1]) - 1:.1%} a year")
assert agrees("P3.6", [math.exp(fl.b[1]) - 1, math.exp(fl.b[0] + 10 * fl.b[1])])
assert agrees("P3.7", [1 - 0.95 ** 20, 1 - 0.95 ** 40])                                      # P3.7
dum = pd.get_dummies(layouts["layout"], dtype=float)[["second", "third"]]                    # P3.8: ANOVA as regression
fl8 = ols(layouts["units"], dum)
print(f"layouts: b = {np.round(fl8.b, 6)}, F = {fl8.F:.2f}, p = {fl8.pF:.4f}; scipy's one-way ANOVA: "
      f"F = {stats.f_oneway(*[layouts.loc[layouts['layout'] == k, 'units'] for k in LAYOUTS]).statistic:.2f}")
assert agrees("P3.8", [*fl8.b, fl8.F])

### The danger of searching: twenty x's of pure noise

Forward selection adds, one at a time, the candidate with the smallest p-value, while it's below 0.05. Give it y and
twenty candidate x's that are all random numbers, with no connection at all. Three of them get in, and the final
model's F test says p ≈ 0.00003. A variable that survives a search has a p-value that is too small.

In [ ]:
rng = random.Random(7)                                # the same random numbers as the course
noise_y = [rng.gauss(0, 1) for _ in range(40)]
noise = pd.DataFrame({f"x{j + 1}": [rng.gauss(0, 1) for _ in range(40)] for j in range(20)})

def forward(y, C, enter=0.05):
    chosen = []
    while True:
        tries = [(ols(y, C[chosen + [c]]).p[-1], c) for c in C if c not in chosen]
        if not tries: return chosen
        p_, c = min(tries)
        if p_ > enter: return chosen
        chosen.append(c)

picked = forward(noise_y, noise)
fn = ols(noise_y, noise[picked])
print(f"forward selection picked {picked} from pure noise: R² = {fn.r2:.2f}, F test p = {fn.pF:.5f}")
assert picked == ["x4", "x11", "x14"] and round(fn.pF, 5) == 0.00003 and round(fn.r2, 2) == 0.48

### Durbin–Watson, and the month-to-month fix

The supplier's residuals run in long stretches of one sign: d ≈ 0.83, below the lower bound 1.273 (n = 24, one x,
α = 0.05), so the errors carry over from month to month. Modelling the *change* in sales on the change in housing
starts gives d ≈ 2.02, close to 2: no autocorrelation left.

In [ ]:
diffs = supplier.diff().dropna()
fd = ols(diffs["sales"], diffs[["starts"]])
print(f"sales on starts: d = {durbin_watson(fs.e):.3f};  changes on changes: d = {durbin_watson(fd.e):.3f}")
if sm is not None: assert abs(sm_durbin_watson(fs.e) - durbin_watson(fs.e)) < 1e-12
assert round(durbin_watson(fs.e), 2) == 0.83 and round(durbin_watson(fd.e), 2) == 2.02

## Tier 4 · The case: Corrowby Fleet Leasing's refurbishments

Corrowby Fleet Leasing leases delivery vans to businesses and sells them at auction when the lease ends (a fictional
company; our own data). A refurbishment that lifts a van from fair condition to good costs **$1,500**. **For which
vans does it pay?** Here are the 240 vans it sold in the last two years, one row each:

| column | what it is |
|---|---|
| `van` | van id, V001–V240 |
| `age` | age at sale, years |
| `km` | distance driven, thousand km |
| `fuel` | diesel, hybrid or electric |
| `condition` | good or fair, from the firm's inspection at sale |
| `price` | auction price, $ |
| `sold_first` | 1 if it sold at its first auction, 0 if it had to be relisted |

The next cell builds the table (the same 240 rows as the spreadsheet's Case study tab) and checks a few of its
numbers.

In [ ]:
# @CASE
# the 240 vans, in van order
CASE = {
    'van': ['V001', 'V002', 'V003', 'V004', 'V005', 'V006', 'V007', 'V008', 'V009', 'V010', 'V011', 'V012',
            'V013', 'V014', 'V015', 'V016', 'V017', 'V018', 'V019', 'V020', 'V021', 'V022', 'V023', 'V024',
            'V025', 'V026', 'V027', 'V028', 'V029', 'V030', 'V031', 'V032', 'V033', 'V034', 'V035', 'V036',
            'V037', 'V038', 'V039', 'V040', 'V041', 'V042', 'V043', 'V044', 'V045', 'V046', 'V047', 'V048',
            'V049', 'V050', 'V051', 'V052', 'V053', 'V054', 'V055', 'V056', 'V057', 'V058', 'V059', 'V060',
            'V061', 'V062', 'V063', 'V064', 'V065', 'V066', 'V067', 'V068', 'V069', 'V070', 'V071', 'V072',
            'V073', 'V074', 'V075', 'V076', 'V077', 'V078', 'V079', 'V080', 'V081', 'V082', 'V083', 'V084',
            'V085', 'V086', 'V087', 'V088', 'V089', 'V090', 'V091', 'V092', 'V093', 'V094', 'V095', 'V096',
            'V097', 'V098', 'V099', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108',
            'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V120',
            'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132',
            'V133', 'V134', 'V135', 'V136', 'V137', 'V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144',
            'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156',
            'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V167', 'V168',
            'V169', 'V170', 'V171', 'V172', 'V173', 'V174', 'V175', 'V176', 'V177', 'V178', 'V179', 'V180',
            'V181', 'V182', 'V183', 'V184', 'V185', 'V186', 'V187', 'V188', 'V189', 'V190', 'V191', 'V192',
            'V193', 'V194', 'V195', 'V196', 'V197', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204',
            'V205', 'V206', 'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216',
            'V217', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226', 'V227', 'V228',
            'V229', 'V230', 'V231', 'V232', 'V233', 'V234', 'V235', 'V236', 'V237', 'V238', 'V239', 'V240'],
    'age': [1.9, 5.2, 6.0, 6.9, 2.4, 2.5, 6.4, 1.8, 6.4, 4.3, 2.1, 4.3, 1.3, 4.3, 4.3, 7.2, 6.3, 4.7, 4.5, 2.2,
            2.7, 5.9, 2.0, 3.1, 1.7, 1.1, 5.4, 2.5, 1.4, 3.0, 6.7, 6.6, 2.3, 2.9, 2.3, 5.4, 4.1, 6.9, 3.2, 1.9,
            6.1, 4.7, 7.0, 5.4, 7.4, 1.7, 2.6, 7.4, 6.9, 7.4, 3.0, 5.2, 3.1, 4.8, 4.6, 4.2, 7.2, 1.7, 6.9, 6.5,
            5.6, 4.6, 2.7, 3.3, 3.6, 5.2, 4.1, 2.8, 2.0, 3.4, 7.2, 5.9, 2.1, 2.7, 5.3, 6.2, 7.2, 2.6, 5.1, 2.8,
            7.3, 3.5, 4.5, 4.7, 3.9, 4.2, 4.9, 6.4, 5.0, 3.6, 4.2, 5.1, 6.3, 4.5, 4.3, 4.3, 2.4, 4.4, 5.6, 2.1,
            5.6, 3.8, 7.3, 4.4, 2.6, 4.2, 2.4, 3.2, 4.2, 4.8, 7.5, 4.8, 6.9, 1.8, 6.5, 5.1, 5.1, 4.8, 3.3, 5.6,
            2.0, 6.3, 3.9, 4.4, 5.8, 3.0, 6.1, 7.4, 6.4, 3.2, 2.9, 2.0, 4.1, 7.0, 6.1, 3.6, 5.5, 1.3, 5.1, 2.1,
            2.8, 5.4, 7.5, 4.2, 6.5, 1.8, 4.7, 3.7, 4.6, 4.7, 6.8, 3.4, 3.3, 4.8, 5.5, 7.3, 1.2, 6.4, 3.3, 5.4,
            6.6, 2.9, 3.5, 7.4, 5.9, 7.0, 6.5, 2.3, 2.0, 3.4, 5.5, 6.3, 1.9, 5.5, 1.0, 5.4, 3.1, 1.5, 1.2, 5.4,
            6.2, 5.4, 6.8, 2.4, 4.1, 4.5, 5.4, 6.5, 3.5, 6.2, 3.5, 3.7, 1.2, 4.3, 6.2, 3.3, 3.3, 3.4, 4.4, 4.2,
            3.5, 7.3, 6.0, 7.0, 1.5, 2.0, 5.1, 6.2, 6.0, 3.8, 5.2, 3.5, 1.1, 7.4, 3.7, 3.3, 5.8, 4.0, 4.5, 3.5,
            5.9, 1.5, 3.3, 1.6, 4.9, 3.1, 5.1, 5.6, 2.7, 4.1, 4.3, 1.6, 2.4, 5.4, 2.2, 7.4, 2.1, 7.4, 5.2, 6.9],
    'km': [71, 138, 180, 184, 58, 67, 137, 29, 157, 107, 22, 117, 16, 149, 139, 226, 136, 130, 95, 20, 88, 153,
           39, 76, 65, 41, 125, 52, 19, 112, 214, 205, 49, 56, 29, 167, 123, 188, 72, 40, 168, 124, 165, 150,
           183, 62, 58, 184, 237, 160, 78, 103, 107, 111, 127, 117, 205, 8, 198, 168, 137, 78, 102, 93, 81, 104,
           110, 69, 57, 78, 175, 185, 54, 118, 130, 149, 209, 68, 100, 58, 165, 89, 153, 112, 84, 96, 158, 136,
           110, 112, 108, 98, 148, 141, 111, 144, 74, 97, 153, 66, 149, 102, 144, 122, 61, 121, 96, 89, 100,
           133, 233, 144, 192, 8, 186, 131, 164, 107, 73, 144, 90, 143, 117, 148, 146, 63, 187, 184, 163, 77,
           64, 69, 150, 186, 148, 34, 99, 56, 124, 58, 96, 90, 194, 72, 155, 32, 97, 83, 133, 107, 215, 88, 82,
           107, 108, 188, 31, 192, 141, 153, 179, 28, 103, 186, 140, 199, 171, 32, 37, 69, 136, 179, 49, 145,
           45, 160, 91, 37, 45, 136, 134, 121, 158, 37, 98, 137, 168, 181, 81, 159, 49, 117, 31, 101, 168, 106,
           46, 77, 145, 90, 98, 168, 154, 189, 54, 93, 128, 209, 207, 113, 119, 62, 47, 187, 107, 102, 145, 111,
           124, 101, 129, 40, 94, 49, 100, 102, 133, 155, 81, 93, 147, 47, 75, 163, 31, 194, 83, 204, 98, 156],
    'fuel': ['diesel', 'diesel', 'hybrid', 'diesel', 'diesel', 'diesel', 'diesel', 'hybrid', 'hybrid', 'diesel',
             'hybrid', 'diesel', 'hybrid', 'diesel', 'diesel', 'electric', 'diesel', 'diesel', 'diesel', 'hybrid',
             'diesel', 'hybrid', 'diesel', 'diesel', 'diesel', 'hybrid', 'electric', 'hybrid', 'electric',
             'diesel', 'electric', 'diesel', 'diesel', 'electric', 'hybrid', 'diesel', 'hybrid', 'hybrid',
             'hybrid', 'diesel', 'hybrid', 'diesel', 'hybrid', 'electric', 'diesel', 'electric', 'hybrid',
             'diesel', 'diesel', 'hybrid', 'diesel', 'diesel', 'diesel', 'diesel', 'hybrid', 'diesel', 'electric',
             'diesel', 'electric', 'diesel', 'diesel', 'diesel', 'diesel', 'diesel', 'diesel', 'hybrid', 'hybrid',
             'diesel', 'diesel', 'hybrid', 'diesel', 'hybrid', 'electric', 'diesel', 'electric', 'diesel',
             'diesel', 'diesel', 'diesel', 'hybrid', 'electric', 'electric', 'electric', 'diesel', 'diesel',
             'diesel', 'diesel', 'electric', 'diesel', 'hybrid', 'hybrid', 'hybrid', 'diesel', 'diesel', 'diesel',
             'diesel', 'diesel', 'hybrid', 'hybrid', 'diesel', 'electric', 'diesel', 'hybrid', 'hybrid', 'diesel',
             'diesel', 'electric', 'hybrid', 'diesel', 'diesel', 'electric', 'electric', 'hybrid', 'electric',
             'diesel', 'electric', 'diesel', 'hybrid', 'electric', 'hybrid', 'electric', 'electric', 'hybrid',
             'diesel', 'hybrid', 'hybrid', 'diesel', 'hybrid', 'diesel', 'electric', 'hybrid', 'diesel',
             'electric', 'diesel', 'diesel', 'diesel', 'diesel', 'diesel', 'diesel', 'diesel', 'electric',
             'hybrid', 'hybrid', 'hybrid', 'diesel', 'hybrid', 'diesel', 'diesel', 'hybrid', 'electric', 'diesel',
             'diesel', 'diesel', 'diesel', 'diesel', 'diesel', 'electric', 'electric', 'diesel', 'diesel',
             'diesel', 'diesel', 'diesel', 'electric', 'diesel', 'diesel', 'hybrid', 'electric', 'hybrid',
             'hybrid', 'diesel', 'diesel', 'diesel', 'hybrid', 'hybrid', 'hybrid', 'hybrid', 'diesel', 'electric',
             'diesel', 'hybrid', 'hybrid', 'diesel', 'diesel', 'diesel', 'hybrid', 'diesel', 'electric', 'hybrid',
             'electric', 'diesel', 'hybrid', 'diesel', 'electric', 'electric', 'electric', 'hybrid', 'diesel',
             'electric', 'electric', 'hybrid', 'diesel', 'diesel', 'diesel', 'hybrid', 'electric', 'diesel',
             'hybrid', 'diesel', 'hybrid', 'hybrid', 'diesel', 'diesel', 'diesel', 'diesel', 'electric', 'diesel',
             'diesel', 'electric', 'diesel', 'hybrid', 'diesel', 'hybrid', 'diesel', 'diesel', 'hybrid', 'diesel',
             'electric', 'diesel', 'hybrid', 'hybrid', 'hybrid', 'electric', 'diesel', 'diesel', 'electric',
             'diesel', 'electric', 'diesel', 'hybrid'],
    'condition': ['good', 'fair', 'fair', 'fair', 'good', 'good', 'fair', 'fair', 'fair', 'fair', 'good', 'fair',
                  'good', 'fair', 'good', 'fair', 'good', 'fair', 'good', 'fair', 'good', 'fair', 'fair', 'good',
                  'good', 'good', 'fair', 'good', 'good', 'good', 'fair', 'fair', 'good', 'fair', 'good', 'fair',
                  'good', 'fair', 'fair', 'good', 'fair', 'fair', 'fair', 'good', 'fair', 'fair', 'good', 'fair',
                  'fair', 'fair', 'fair', 'fair', 'good', 'good', 'good', 'fair', 'fair', 'fair', 'fair', 'fair',
                  'fair', 'fair', 'fair', 'fair', 'fair', 'good', 'fair', 'good', 'fair', 'fair', 'fair', 'fair',
                  'fair', 'fair', 'fair', 'good', 'good', 'good', 'fair', 'fair', 'fair', 'good', 'good', 'fair',
                  'fair', 'good', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair',
                  'good', 'good', 'fair', 'good', 'fair', 'fair', 'fair', 'fair', 'good', 'fair', 'fair', 'good',
                  'fair', 'fair', 'fair', 'fair', 'good', 'good', 'good', 'fair', 'good', 'fair', 'good', 'fair',
                  'good', 'good', 'fair', 'good', 'fair', 'good', 'fair', 'fair', 'fair', 'fair', 'good', 'good',
                  'fair', 'fair', 'fair', 'good', 'fair', 'good', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair',
                  'fair', 'fair', 'fair', 'fair', 'fair', 'fair', 'fair', 'good', 'good', 'fair', 'fair', 'fair',
                  'good', 'fair', 'fair', 'fair', 'fair', 'fair', 'good', 'fair', 'fair', 'fair', 'fair', 'good',
                  'good', 'good', 'fair', 'fair', 'fair', 'fair', 'good', 'fair', 'fair', 'good', 'good', 'fair',
                  'fair', 'fair', 'fair', 'good', 'fair', 'good', 'fair', 'good', 'good', 'fair', 'fair', 'good',
                  'good', 'fair', 'fair', 'good', 'good', 'fair', 'good', 'good', 'good', 'fair', 'fair', 'good',
                  'fair', 'good', 'fair', 'fair', 'fair', 'fair', 'good', 'fair', 'good', 'fair', 'good', 'fair',
                  'fair', 'good', 'fair', 'fair', 'good', 'good', 'good', 'good', 'good', 'good', 'good', 'good',
                  'fair', 'fair', 'fair', 'good', 'fair', 'fair', 'good', 'fair', 'good', 'fair', 'fair', 'fair'],
    'price': [26160, 16560, 15780, 13570, 28790, 29690, 14440, 33980, 14500, 19480, 30990, 17510, 27800, 18440,
              20270, 13680, 16670, 20330, 19840, 26500, 23760, 17190, 27310, 25910, 28650, 35480, 18390, 27760,
              45690, 23660, 15170, 13310, 26750, 26770, 39650, 13620, 22850, 16390, 29080, 30360, 14710, 19190,
              14650, 20570, 13080, 28950, 29480, 14370, 12120, 13340, 24460, 15180, 21290, 19770, 21990, 18950,
              16010, 29040, 14770, 12540, 15540, 18890, 24140, 23210, 21070, 21940, 23150, 26320, 22410, 20610,
              12900, 16550, 28560, 20620, 17140, 17200, 15240, 27330, 16400, 25050, 14060, 27770, 21360, 20710,
              21710, 20830, 15490, 17940, 17910, 20320, 18420, 16400, 14320, 21290, 17870, 18030, 33510, 24840,
              15930, 24490, 17460, 17910, 15280, 20920, 24140, 20290, 27790, 25440, 20890, 20790, 13670, 18700,
              15630, 36810, 15870, 16750, 18200, 21680, 28620, 14890, 28760, 18650, 22860, 22390, 17530, 28750,
              14240, 11370, 15410, 24950, 30010, 28210, 18820, 12740, 13360, 27170, 16190, 34440, 17600, 26270,
              23890, 17670, 13960, 24310, 14840, 35690, 18450, 20500, 19300, 22090, 12170, 22220, 25300, 25350,
              18400, 12380, 40740, 16530, 19540, 17350, 12320, 25060, 23130, 15460, 17380, 12950, 16200, 31090,
              33340, 26890, 16590, 13820, 28630, 18430, 35840, 17110, 25430, 30960, 38750, 18790, 14240, 16750,
              12960, 30580, 20880, 19190, 19170, 16040, 26260, 16340, 21520, 23310, 33490, 23000, 15310, 30390,
              21470, 26480, 23620, 27730, 24110, 11560, 14590, 14060, 34600, 30340, 14450, 14570, 13260, 17760,
              20150, 21690, 32610, 11810, 25190, 26650, 16680, 23300, 22650, 24440, 20520, 35120, 24610, 29790,
              18630, 21360, 20580, 18020, 24300, 20580, 16410, 31810, 25010, 16760, 29890, 13390, 33440, 12640,
              20380, 14890],
    'sold_first': [1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                   1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1,
                   0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1,
                   1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1,
                   0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0,
                   0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1,
                   1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0,
                   1, 1],
}
# @END CASE
case = pd.DataFrame(CASE)

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'n': 240,
 'n_good': 88,
 'n_fair': 152,
 'mean_good': 26154.772727272728,
 'mean_fair': 18714.934210526317,
 'r2': 0.9122776746590543,
 'b': [10.481864982279056,
       -0.10360704251408359,
       -0.0014981996331915673,
       0.05153189666304392,
       0.11658605297642377,
       0.10157104781127302],
 'pct_good': 0.10690855902867846,
 'good_ci': [0.07818101813265432, 0.13640152946951334],
 'breakeven': 14030.682048549748,
 'breakeven_low': 19186.242848038408,
 'fair_above': 129,
 'fair_above_low': 59,
 'logit_b': [2.533248116663548, -0.3873718737300296, 0.6503762511809387]}
# @END STATS
S = CASE_STATS
assert len(case) == S["n"] == 240 and (case["condition"] == "good").sum() == S["n_good"]
assert abs(case.loc[case["condition"] == "good", "price"].mean() - S["mean_good"]) < 1e-9
print(f"{len(case)} vans, {case.shape[1]} columns: the same data as the course's case.")
case.head(8)

### How you'd do this in Python

Dummies for fuel (diesel is the baseline) and for condition (fair is the baseline), a column of `np.log(price)`, and
each step of the case is a line or two. The cells below reproduce the video's case: the tempting comparison, why the
log model fits better than the straight one, the premium for good condition with its interval, the checks, one van's
prediction, and the break-even price.

In [ ]:
case = case.assign(hybrid=(case["fuel"] == "hybrid").astype(float), electric=(case["fuel"] == "electric").astype(float),
                   good=(case["condition"] == "good").astype(float), ln_price=np.log(case["price"]))
XS = ["age", "km", "hybrid", "electric", "good"]
means = case.groupby("condition")[["price", "age"]].mean()                                   # 1 · the tempting answer
print(f"1 · good vans average ${means.loc['good', 'price']:,.0f}, fair ${means.loc['fair', 'price']:,.0f}: a gap of "
      f"${means.loc['good', 'price'] - means.loc['fair', 'price']:,.0f}. But good vans are younger: "
      f"{means.loc['good', 'age']:.1f} years against {means.loc['fair', 'age']:.1f}.")
lin = ols(case["price"], case[XS]); full = ols(case["ln_price"], case[XS])                   # 2 · straight or log?
def thirds(f):
    o = np.argsort(f.yhat); return [np.std(f.e[o][j * 80:(j + 1) * 80], ddof=1) for j in range(3)]
print(f"2 · the straight model's residuals fan out (sd by thirds of ŷ: {', '.join(f'${v:,.0f}' for v in thirds(lin))}); "
      f"the log model's stay even ({', '.join(f'{v:.3f}' for v in thirds(full))}); R² {full.r2:.3f}")
pct = np.exp(full.b[1:]) - 1                                                                 # 3 · percents
q = stats.t.ppf(0.975, full.dfe)
good_ci = np.exp(full.b[5] + np.array([-1, 1]) * q * full.se[5]) - 1
print("3 · " + "; ".join(f"{nm} {v:+.2%}" for nm, v in zip(["age (a year)", "km (a thousand)", "hybrid", "electric",
                                                            "good"], pct))
      + f"; good's 95% interval {good_ci[0]:.1%} to {good_ci[1]:.1%}")
vifs = [1 / (1 - ols(case[c], case[[o for o in XS if o != c]]).r2) for c in XS]              # 4 · checks
red = ols(case["ln_price"], case[["age", "km", "good"]])
F_fuel = ((red.sse - full.sse) / 2) / (full.sse / full.dfe)
print(f"4 · r(age, km) {case['age'].corr(case['km']):.2f}; VIFs {', '.join(f'{v:.1f}' for v in vifs)}; fuel's partial "
      f"F {F_fuel:.1f} (p {stats.f.sf(F_fuel, 2, full.dfe):.1e}); {(np.abs(full.std) > 2).sum()} of 240 beyond ±2")
yh, _, (p0, p1), _, _ = intervals(full, [3, 80, 0, 0, 0])                                     # 5 · one van
print(f"5 · a 3-year-old diesel, 80,000 km, fair: ${math.exp(yh):,.0f} (95% prediction interval ${math.exp(p0):,.0f} "
      f"to ${math.exp(p1):,.0f})")
fair = case["good"] == 0                                                                     # 6 · the decision
fair_pred = np.exp(full.yhat[fair.to_numpy()])
be, be_low = 1500 / pct[4], 1500 / good_ci[0]
print(f"6 · the refurbishment pays above ${be:,.0f} (at the premium's estimate) or ${be_low:,.0f} (at its low end): "
      f"{(fair_pred > be).sum()} and {(fair_pred > be_low).sum()} of the {fair.sum()} fair vans")
assert round(means.loc["good", "price"]) == round(S["mean_good"]) and round(full.r2, 3) == round(S["r2"], 3)
assert np.allclose(full.b, S["b"]) and abs(pct[4] - S["pct_good"]) < 1e-12 and np.allclose(good_ci, S["good_ci"])
assert round(be, -1) == 14030 and round(be_low, -1) == 19190 and (fair_pred > be_low).sum() == S["fair_above_low"] == 59
if sm is not None:
    sm_case = sm.OLS(case["ln_price"], sm.add_constant(case[XS])).fit()
    assert np.allclose(sm_case.params, full.b) and np.allclose(sm_case.bse, full.se)

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.2))
fig.subplots_adjust(left=0.06, right=0.98, top=0.82, bottom=0.16)
ys = np.random.default_rng(1).uniform(-1, 1, len(fair_pred))
ax.scatter(fair_pred, ys, s=26, color=np.where(fair_pred > be_low, GOLD, np.where(fair_pred > be, CHALK, DIM)), zorder=3)
for x, lab, col in ((be, f"${round(be, -1):,.0f}: pays at the estimate", DIM), (be_low, f"${round(be_low, -1):,.0f}: pays at the low end", GOLD)):
    ax.axvline(x, color=col, ls="--", lw=1.6); ax.text(x + 150, 1.25, lab, color=col, fontsize=11)
ax.set_ylim(-1.5, 1.6); ax.set_yticks([]); ax.set_xlabel("predicted auction price as a fair van ($)")
fig.suptitle(f"{(fair_pred > be_low).sum()} of {fair.sum()} fair vans clear the low-end break-even", x=0.06, ha="left",
             fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the fuel test, the two correlated x's, a second question (which
vans sell at their first auction? a logistic regression on age and condition), ending in your recommendation. Use the
DataFrame `case` in a cell of your own (press **+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `case` with pandas, numpy and scipy and checks
it against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
mine = {}
mine["P4.1"] = [means.loc["good", "price"], means.loc["fair", "price"], means.loc["good", "price"] - means.loc["fair", "price"]]
mine["P4.2"] = [*full.b[1:], full.r2]
mine["P4.3"] = [pct[4], *good_ci]
mine["P4.4"] = [math.exp(yh), math.exp(p0), math.exp(p1)]
mine["P4.5"] = [F_fuel, stats.f.sf(F_fuel, 2, full.dfe)]
vif6 = np.diag(full.inv)[1:3] * [((case[c] - case[c].mean()) ** 2).sum() for c in ("age", "km")]   # VIF from (X′X)⁻¹
mine["P4.6"] = [case["age"].corr(case["km"]), *vif6, ols(case["ln_price"], case[["km", "hybrid", "electric", "good"]]).b[1]]
lgc = logit(case["sold_first"], case[["age", "good"]])
mine["P4.7"] = [math.exp(lgc.b[1]), math.exp(lgc.b[2]), prob(lgc.b[0] + 3 * lgc.b[1] + lgc.b[2])]
mine["P4.8"] = [be_low, (fair_pred > be_low).sum()]

assert np.allclose(vif6, vifs[:2])
if sm is not None:
    sm_lg = sm.Logit(case["sold_first"], sm.add_constant(case[["age", "good"]])).fit(disp=0)
    assert np.allclose(sm_lg.params, lgc.b)
for k, got in mine.items():
    got = [float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(v, 6)) if abs(v) >= 1e-4 else f'{v:.2e}'}" for p, v in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")
print(f"Good condition multiplies the odds of selling first time by {math.exp(lgc.b[2]):.2f} (z = {lgc.z[2]:.2f}, "
      f"p = {lgc.pz[2]:.2f}: not clear at 0.05).")

## Sources
Every exercise, data set and company here is our own (Corrowby Fleet Leasing is fictional; the name was searched,
2026-11-09). The chapter order follows a standard first course in business statistics (multiple regression and
model building); the coverage was cross-checked with OpenStax, Introductory Business Statistics 2e, chapter 13
(retrieved 2026-11-09). The Durbin–Watson bounds are Savin and White's (Econometrica 45(8), 1977), as tabulated by
the University of York (retrieved 2026-11-09). The leverage rule 3(p + 1)/n and the Cook's distance rule (above 1)
are common conventions.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs07_multiple_regression/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs07_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert all(abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))) and \
            len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    for _k in ("age", "km", "fuel", "condition", "price", "sold_first", "van"):
        assert _P.CASE[_k] == CASE[_k], _k
    for _name, _d in (("RENT", FLATS), ("LOANS", LOANS), ("FUEL", ROUTES), ("HEAT", WAREHOUSES), ("FORK", FORKLIFTS),
                      ("SUPPLY", SUPPLIER)):
        assert all(getattr(_P, _name)[k] == _d[k] for k in getattr(_P, _name)), _name
    assert _P.LAYOUT == LAYOUTS and _P.NOISE_Y == noise_y
    print("matches the course's practice set")